# Lesson 8: Urban Heat Island

**Planning question: Which neighbourhoods are hottest?**

Cities are hotter than the countryside around them because concrete and asphalt soak up heat and there are fewer trees.
This is the **urban heat island**. Heat is a growing health risk in Indian cities, especially before the monsoon.

In this notebook you will map **land surface temperature (LST)** across Bengaluru from Landsat's thermal sensor,
find the hottest wards, and test how much cooler greener wards are.

Part of [RS & GIS for Spatial Planners](https://soniadas123.github.io/rs-gis-spatial-planner/).

In [ ]:
!pip install -q pystac-client planetary-computer odc-stac rioxarray

In [ ]:
import pystac_client
import planetary_computer
import odc.stac
import rioxarray
import geopandas as gpd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rasterio.features import rasterize
from shapely.geometry import shape, box

## Settings

We use the hot, dry pre-monsoon months of March to May.

In [ ]:
WARDS_URL = "https://raw.githubusercontent.com/datameet/Municipal_Spatial_Data/master/Bangalore/BBMP.geojson"
DATES = "2024-03-01/2024-05-31"
UTM_CRS = "EPSG:32643"

wards = gpd.read_file(WARDS_URL)
BBOX = list(wards.total_bounds)

## Step 1: Find a clear Landsat image

**Landsat 8 and 9** are NASA and USGS satellites with a **thermal infrared** sensor, which measures heat given off by the ground.
The thermal band (`lwir11`) has 100 m pixels, which are delivered resampled to 30 m.

We pick the least cloudy image that covers all of Bengaluru.

In [ ]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)
items = catalog.search(
    collections=["landsat-c2-l2"],
    bbox=BBOX,
    datetime=DATES,
    query={"eo:cloud_cover": {"lt": 10}, "platform": {"in": ["landsat-8", "landsat-9"]}},
).item_collection()

city_box = box(*BBOX)


def coverage(item):
    """Share of the city box covered by this image (0 to 1)."""
    return shape(item.geometry).intersection(city_box).area / city_box.area


# Best coverage first, then least cloud
item = min(items, key=lambda i: (-round(coverage(i), 2), i.properties["eo:cloud_cover"]))
print("Images found:", len(items))
print("Using:", item.id, item.datetime.date(),
      f"cloud {item.properties['eo:cloud_cover']}%, covers {coverage(item) * 100:.0f}% of the city")

## Step 2: Load the bands

- `lwir11`: thermal band, for temperature
- `red` and `nir08`: for NDVI
- `qa_pixel`: quality flags that mark clouds

In [ ]:
ds = odc.stac.load([item], bands=["lwir11", "red", "nir08", "qa_pixel"],
                   bbox=BBOX, crs=UTM_CRS, resolution=30).isel(time=0)
ds

## Step 3: Convert to degrees Celsius

The thermal values are stored as whole numbers. The USGS gives the formula to convert them to temperature in Kelvin:

`Kelvin = value x 0.00341802 + 149.0`, and then `Celsius = Kelvin - 273.15`.

We also remove cloudy pixels. In the `qa_pixel` band, bit 1 marks dilated cloud, bit 3 marks cloud and bit 4 marks cloud shadow.

In [ ]:
cloud_bits = 0b11010   # bits 1, 3 and 4
clear = ((ds["qa_pixel"] & cloud_bits) == 0) & (ds["lwir11"] > 0)

lst = (ds["lwir11"].astype("float32") * 0.00341802 + 149.0 - 273.15).where(clear)
print(f"Temperature range: {float(lst.quantile(0.01)):.1f} to {float(lst.quantile(0.99)):.1f} C")

## Step 4: Map land surface temperature

Remember that this is the temperature of the **ground surface** at about 10:30 in the morning, when Landsat passes over.
Roofs and roads can be much hotter than the air.

In [ ]:
wards_utm = wards.to_crs(UTM_CRS).reset_index(drop=True)

lst.plot(cmap="inferno", robust=True, size=9, cbar_kwargs={"label": "Land surface temperature (C)"})
wards_utm.boundary.plot(ax=plt.gca(), color="white", linewidth=0.2)
plt.title(f"Land surface temperature, {item.datetime.date()}")
plt.show()

## Step 5: NDVI from the same image

Landsat surface reflectance uses the formula `reflectance = value x 0.0000275 - 0.2`.

In [ ]:
red = (ds["red"].astype("float32") * 0.0000275 - 0.2).where(clear)
nir = (ds["nir08"].astype("float32") * 0.0000275 - 0.2).where(clear)
ndvi = (nir - red) / (nir + red)

## Step 6: How strong is the heat island?

We compare the average temperature of built-up pixels (NDVI below 0.2) with that of densely vegetated pixels (NDVI above 0.4).

In [ ]:
built_temp = float(lst.where(ndvi < 0.2).mean())
green_temp = float(lst.where(ndvi > 0.4).mean())
print(f"Built-up surfaces:  {built_temp:.1f} C")
print(f"Dense vegetation:   {green_temp:.1f} C")
print(f"Difference:         {built_temp - green_temp:.1f} C")

## Step 7: Hot spots

A **hot spot** is a pixel that is much hotter than the city average. Here, that means more than 1.5 standard deviations above the mean.

In [ ]:
threshold = float(lst.mean() + 1.5 * lst.std())
hotspots = lst.where(lst > threshold)

fig, ax = plt.subplots(figsize=(9, 9))
lst.plot(ax=ax, cmap="Greys", add_colorbar=False, robust=True)
hotspots.plot(ax=ax, cmap="autumn_r", add_colorbar=False)
wards_utm.boundary.plot(ax=ax, color="black", linewidth=0.2)
ax.set_title(f"Hot spots (above {threshold:.1f} C) in orange-red")
ax.set_axis_off()
plt.show()

## Step 8: Average temperature per ward

We use the same rasterize-and-group method as in Lesson 7.

In [ ]:
wards_utm["zone"] = wards_utm.index + 1
zones = rasterize(zip(wards_utm.geometry, wards_utm["zone"]), out_shape=lst.shape,
                  transform=lst.rio.transform(), fill=0, dtype="int32")

pixels = pd.DataFrame({"zone": zones.ravel(), "lst": lst.values.ravel(), "ndvi": ndvi.values.ravel()}).dropna()
ward_stats = pixels[pixels["zone"] > 0].groupby("zone")[["lst", "ndvi"]].mean()
wards_utm = wards_utm.merge(ward_stats, left_on="zone", right_index=True, how="left")

ax = wards_utm.plot(column="lst", cmap="inferno", legend=True, figsize=(9, 9), edgecolor="grey", linewidth=0.2,
                    legend_kwds={"label": "Mean land surface temperature (C)", "shrink": 0.6})
ax.set_title("Hottest and coolest wards")
ax.set_axis_off()
plt.show()

In [ ]:
print("Hottest wards:")
wards_utm.nlargest(10, "lst")[["KGISWardName", "lst", "ndvi"]].round(2)

## Step 9: Do greener wards stay cooler?

Each dot is a ward. The line shows the trend.

In [ ]:
valid = wards_utm.dropna(subset=["lst", "ndvi"])
slope, intercept = np.polyfit(valid["ndvi"], valid["lst"], 1)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(valid["ndvi"], valid["lst"], s=12, color="darkorange")
x = np.linspace(valid["ndvi"].min(), valid["ndvi"].max(), 50)
ax.plot(x, slope * x + intercept, color="black")
ax.set_xlabel("Mean NDVI (greener to the right)")
ax.set_ylabel("Mean land surface temperature (C)")
ax.set_title("Greener wards are cooler")
plt.show()

print(f"On average, each +0.1 in ward NDVI goes with {slope * 0.1:.1f} C in surface temperature.")

## Step 10: Save the results

In [ ]:
lst.rio.to_raster("bengaluru_lst.tif")
result = wards_utm[["KGISWardName", "lst", "ndvi", "geometry"]].round(2)
result.to_file("bengaluru_heat.gpkg", driver="GPKG")
print("Saved bengaluru_heat.gpkg and bengaluru_lst.tif")

## What this means for planning

- Heat maps show where heat action plans, cool roofs and tree planting are needed most.
- The link between NDVI and temperature makes the case for green infrastructure as a cooling strategy.
- Hot spots often match industrial roofs, bus depots, large parking areas and bare construction sites.

**Limitations:** this is surface temperature at one moment on one day, not air temperature.
Thermal pixels are about 100 m across, so small features blur together. For robust results, repeat the analysis with several dates.

## Try it yourself

1. Change `DATES` to `"2015-03-01/2015-05-31"` (Landsat 8 only). Has the city got hotter?
2. Change the hot spot threshold from 1.5 to 2 standard deviations.
3. Look up the hottest ward on Google Maps satellite view. What kind of land use do you see?

---
*Enjoying the course? Support it with a [star on GitHub](https://github.com/soniadas123/rs-gis-spatial-planner).*